# step01_suna_spectral_qc - SUNA Stage 1: per-frame spectral QC (P45_06)

Stage 0 asked "what is in each file and is the instrument healthy". Stage 1 asks the next question, frame by
frame: **is this spectrum trustworthy enough to turn into a nitrate value?** It takes the raw frames, builds
the per-frame absorbance, runs a set of physically motivated spectral checks, and writes a Level-1 table where
every frame is either clean or carries a reason it is not.

**Why spectral QC is the right second step.** A SUNA nitrate is the output of a least-squares fit of the
217 to 240 nm absorbance to reference spectra for nitrate, bromide and a linear baseline (Johnson and Coletti,
2002; Sakamoto, Johnson and Coletti, 2009). The fit will always return a number, even from a spectrum taken in
air, through a bubble, or with a fouled window. So a plausible-looking nitrate is not evidence the measurement
was good. Stage 1 checks the spectrum itself, which is the only honest way to know (Johnson et al., 2018).

**What Stage 1 does NOT do (on purpose).** It does not split the continuous logs into casts and it does not
assign depth. From Stage 0 we learned the P45_06 files are continuous across several casts, with the instrument
often left logging between them (on deck as well as in the water), so there are no clean gaps to cut on. Splitting and depth both need the CTD
time-and-pressure record, which comes in at co-location. Until then Stage 1 only *flags*; it deletes nothing, so
no frame is lost before we have the CTD timebase. The long between-cast baseline stretches are kept and simply
marked, not removed.

**The checks, and the reasoning behind each (thresholds derived from your own P45_06 data, not guessed):**
- **out of water** (optical gate): in seawater the lamp light is attenuated to a stable level and the baseline
  absorbance sits near a fixed value; in air or through a bubble it is not. A frame is flagged when its peak
  counts, `abs254` or `spec_avg` falls outside the in-water window (median +- 6 robust sigma of in-water frames,
  see below). This catches the recovery and surface-break frames, and the deck frames between casts.
- **saturation in the fit window**: any 217 to 240 nm channel at the 16-bit ceiling makes the fit meaningless.
- **low signal / high RMSE / warm-up / humidity**: carried from Stage 0 (lamp or optics problems, poor fits).
- **long-wave baseline anomaly**: nitrate and bromide do not absorb above ~310 nm, so clean water shows near-zero
  absorbance there. Mean `|absorbance|` over 310 to 360 nm above 0.10 means turbidity, CDOM or a bubble has
  tilted the baseline, which biases the fit.
- **shape anomaly**: a frame whose spectrum shape (250 to 360 nm, OUTSIDE the nitrate band so real nitrate does
  not trip it) departs from the clean reference by a correlation drop above 0.01 has a broadband distortion
  (bubble, fouling). Measuring this outside the fit window is deliberate: a within-window shape test would reject
  real high-nitrate frames, which is the opposite of what we want.

**Where "in water" comes from (Nov 2026).** The out-of-water window and the reference spectrum (against which the
long-wave baseline and the shape are measured) both need a population of frames known to be in seawater. They
used to come from each file's own frames, which works only when most of a file is in water. On P45_06 the SUNA
often logged for hours on deck between casts with a wet optical path (peak about 38,000 counts against about
45,000 in seawater), so in several files the deck frames were the majority: real seawater frames were then
flagged out of water or as a baseline anomaly (casts 09 and 10 kept 68 and 5 usable frames). Now both come from
frames taken while the CTD was in the water: inside each CTD cast's wet interval (conductivity above 1) shrunk by
`OOW_CTD_MARGIN_S` (60 s) at both ends, wider than any plausible SUNA clock error, so no clock correction is
needed here. A file with fewer than `OOW_MIN_N_CTD` such frames, or a cruise with no CTD record, keeps the file's
own population. A window taken from seawater is bounded on the dark side only (air, a wet path on deck and
bubbles all read darker, with higher apparent absorbance; seawater frames that read brighter are not out of water)
and is never narrower than `OOW_DARK_FRAC` (10 %) of the median for peak counts and `spec_avg`, or `OOW_ABS254_TOL`
(0.05) for `abs254`: on both cruises the deck state sat 15 % darker and 0.08 higher in `abs254`, while near-bottom
turbidity and the optical path settling during the soak moved seawater frames by up to about 9 %. Separately,
**`f_offcast`** marks frames more than 60 s from every CTD cast (deck time, deck tests): they have no CTD record to
place them, so they are not clean, but `f_outofwater` stays purely optical so step04's water-entry check still
reads the instrument. The run prints, per file and per cast, the clean counts under the old and new rules, and
saves them (`_audit/suna/<CRUISE>__suna_oow_gate_comparison*.csv`). Code: `suncat.nitrate.spectral_qc`.

A frame is **L1 clean** when none of the fatal flags fire (humidity is advisory). The interactive explorer lets
you step to the next flagged frame and see exactly why it failed: raw spectrum, absorbance near the fit window,
and the long-wave baseline.

Sources: Johnson and Coletti (2002), *DSR I*; Sakamoto, Johnson and Coletti (2009), *L&O: Methods*;
Johnson et al. (2018), *JTECH* (BGC-Argo nitrate QC and spectral baseline checks); Sea-Bird SUNA V2 manual.

In [ ]:
import re, sys
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib

INTERACTIVE_ZOOM = True
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try:
    from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image = None

# resolve repo root
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, find_suna_cal, product_dir
try:
    from suncat.nitrate.spectral_qc import QCConfig, qc_frames as _qc_frames, ctd_wet_intervals, in_cast, off_cast, which_cast
except ImportError:                                   # suncat not installed: use the copy in the repo
    sys.path.insert(0, str(Path(_p) / "src"))
    from suncat.nitrate.spectral_qc import QCConfig, qc_frames as _qc_frames, ctd_wet_intervals, in_cast, off_cast, which_cast

# config
CRUISE_ID  = "P45_07"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
SUNA_L0    = product_dir(CRUISE_ID, "suna_raw")
CAL_FILE   = find_suna_cal(cruise_id=CRUISE_ID, filename="SNA2503C.cal")
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L1_DIR     = product_dir(CRUISE_ID, "suna_l1");     L1_DIR.mkdir(parents=True, exist_ok=True)

NSPEC = 256; SPEC_START = 11
FIT_LO, FIT_HI = 217.0, 240.0

SAT_COUNTS = 60000; LOW_COUNTS = 2000; RMSE_HI = 1.0e-3; HUM_HI = 10.0; WARMUP_S = 60.0

# ---- optical out-of-water gate ---------------------------------------------
# abs254 and spec_avg are computed ONBOARD with the instrument's active cal, so
# their in-water level shifts between lamp/cal regimes (P45_06 vs P45_07 G.cal).
# The fixed windows below are the P45_06 values, kept ONLY as a fallback. With
# OOW_ADAPTIVE=True the gate is re-derived per file from that file's own in-water
# population (frames that pass every OTHER fatal check), with hard physical
# guards, so it tracks the lamp regime. The derived window is printed and saved;
# freeze it to fixed values later if Pierre prefers fixed thresholds.
OOW_ADAPTIVE = True
OOW_K        = 6.0      # half-width of the derived window, in robust (MAD) sigma
OOW_MIN_N    = 50       # need this many good frames to derive, else use fallback
PEAK_LO, PEAK_HI       = 20000, 58000     # fallback (P45_06)
ABS254_LO, ABS254_HI   = -0.60, -0.15     # fallback (P45_06)
SPECAVG_LO, SPECAVG_HI = 8000, 30000      # fallback (P45_06)
PEAK_HARD    = (LOW_COUNTS, SAT_COUNTS)   # derived window never exceeds these
ABS254_HARD  = (-5.0, 5.0)
SPECAVG_HARD = (0.0, SAT_COUNTS)
# In-water population from the CTD (suncat.nitrate.spectral_qc): frames inside a CTD cast's wet interval shrunk by
# OOW_CTD_MARGIN_S at both ends (wider than any plausible SUNA clock error) define the window and the reference
# spectrum. Files with fewer than OOW_MIN_N_CTD such frames keep the file's own population (the rule before Nov 2026).
OOW_FROM_CTD     = True
OOW_CTD_MARGIN_S = 60.0
OOW_MIN_N_CTD    = 30
OOW_DARK_FRAC    = 0.10   # CTD-derived window: the dark-side bound is at least 10 % below the median (peak, spec_avg)
OOW_ABS254_TOL   = 0.05   # ... and at least 0.05 above the median abs254
CTD_CNV_DIR      = product_dir(CRUISE_ID, "ctd_datcnv")

LONGWAVE_LO, LONGWAVE_HI = 310.0, 360.0
LONGWAVE_A_HI          = 0.10
SHAPE_LO, SHAPE_HI     = 250.0, 360.0
SHAPE_DEV_HI           = 0.01

EXPLORER_FIGSIZE = (13.5, 5.2)
print("cruise:", CRUISE_ID, "| L0:", SUNA_L0, "| L1 out:", L1_DIR)
print("out-of-water gate:", (("ADAPTIVE, k=%.1f, in-water population " % OOW_K)
       + (f"from the CTD (wet interval minus {OOW_CTD_MARGIN_S:g} s at each end), else the file's own frames"
          if OOW_FROM_CTD else "from each file's own frames")) if OOW_ADAPTIVE else "FIXED P45_06 windows")

In [ ]:
# readers (standalone, same as Stage 0)
def parse_cal(path):
    wl = []
    try: lines = Path(path).read_text(errors="replace").splitlines()
    except Exception as e:
        print("cal read failed:", e); return np.arange(NSPEC, dtype=float)
    for ln in lines:
        p = ln.split(",")
        if p[0] == "E" and len(p) >= 2:
            try: wl.append(float(p[1]))
            except ValueError: pass
    return np.array(wl, float)

def _ts(datef, timed):
    y = int(float(datef)) // 1000; doy = int(float(datef)) % 1000
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=float(timed))

def parse_suna_file(path):
    fhr, rows, spec = {}, [], []
    for ln in Path(path).read_text(errors="replace").splitlines():
        if ln.startswith("SATFHR"):
            p = ln.split(",", 2)
            if len(p) == 3: fhr[p[1].strip()] = p[2].strip()
        elif ln.startswith("SATSLF"):
            f = ln.split(",")
            if len(f) < SPEC_START + NSPEC + 6: continue
            try: sp = np.array(f[SPEC_START:SPEC_START + NSPEC], float)
            except ValueError: continue
            tr = f[SPEC_START + NSPEC:]
            def gg(i):
                try: return float(tr[i])
                except (ValueError, IndexError): return np.nan
            try:
                rows.append(dict(time_utc=_ts(f[1], f[2]),
                    nitrate_uM=float(f[3]), abs254=float(f[5]), abs350=float(f[6]),
                    bromide=float(f[7]), spec_avg=float(f[8]), dark=float(f[9]),
                    T_int=gg(0), T_lamp=gg(2), lamp_s=gg(3), humidity=gg(4),
                    I_main=gg(8), fit_rmse=gg(13)))
                spec.append(sp)
            except ValueError: continue
    return fhr, pd.DataFrame(rows), (np.array(spec) if spec else np.empty((0, NSPEC)))

CFG = QCConfig(sat_counts=SAT_COUNTS, low_counts=LOW_COUNTS, rmse_hi=RMSE_HI, hum_hi=HUM_HI, warmup_s=WARMUP_S,
               fit_lo=FIT_LO, fit_hi=FIT_HI, longwave=(LONGWAVE_LO, LONGWAVE_HI), longwave_a_hi=LONGWAVE_A_HI,
               shape=(SHAPE_LO, SHAPE_HI), shape_dev_hi=SHAPE_DEV_HI, oow_adaptive=OOW_ADAPTIVE, oow_k=OOW_K,
               oow_min_n=OOW_MIN_N, oow_min_n_ctd=OOW_MIN_N_CTD, oow_dark_frac=OOW_DARK_FRAC,
               oow_abs254_tol=OOW_ABS254_TOL, fallback_peak=(PEAK_LO, PEAK_HI),
               fallback_abs254=(ABS254_LO, ABS254_HI), fallback_specavg=(SPECAVG_LO, SPECAVG_HI),
               peak_hard=PEAK_HARD, abs254_hard=ABS254_HARD, specavg_hard=SPECAVG_HARD)


def qc_frames(df, spec, wl, incast=None, offcast=None):
    """L0 integrity + L1 spectral QC (suncat.nitrate.spectral_qc). incast: frames taken while the CTD was in the
    water, which then define the reference spectrum and the out-of-water window; None = the file's own frames.
    offcast: frames more than OOW_CTD_MARGIN_S from every CTD cast (f_offcast)."""
    return _qc_frames(df, spec, wl, CFG, incast, offcast)


FATAL_FLAGS = ["f_outofwater","f_satfit","f_saturated","f_lowsignal",
               "f_highrmse","f_baseline","f_shape","f_warmup","f_offcast"]
ALL_FLAGS = FATAL_FLAGS + ["f_humid"]

In [ ]:
# run Stage 1 QC
CAL_WL = parse_cal(CAL_FILE)
if CAL_WL.size != NSPEC:
    print(f"cal has {CAL_WL.size} wl (expected {NSPEC}); using channel index"); CAL_WL = np.arange(NSPEC, dtype=float)
files = sorted(set(list(SUNA_L0.glob("*.CSV")) + list(SUNA_L0.glob("*.csv"))))
INTERVALS = pd.DataFrame(columns=["cast", "t_in", "t_out"])
if OOW_ADAPTIVE and OOW_FROM_CTD:
    _cnvs = sorted(CTD_CNV_DIR.glob("*.cnv")) if CTD_CNV_DIR.exists() else []
    INTERVALS, _skipped = ctd_wet_intervals(_cnvs)
    print(f"CTD in the water: {len(INTERVALS)} cast(s) from {len(_cnvs)} .cnv in {CTD_CNV_DIR}"
          + (f" (by {', '.join(sorted(set(INTERVALS['by'])))})" if len(INTERVALS) else "")
          + (f"; left out: {_skipped}" if _skipped else ""))
    if INTERVALS.empty:
        print("  no CTD wet intervals: every file uses its own frames (the rule before Nov 2026)")
FRAMES, SPECTRA, REFS, inv, oow_rows, cmp_rows, cast_rows = {}, {}, {}, [], [], [], []
for p in files:
    fhr, df, sp = parse_suna_file(p)
    if df.empty: print("  ", p.name, "no frames"); continue
    t_ep = (df["time_utc"] - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
    inc = in_cast(t_ep, INTERVALS, OOW_CTD_MARGIN_S) if len(INTERVALS) else None
    off = off_cast(t_ep, INTERVALS, OOW_CTD_MARGIN_S) if len(INTERVALS) else None
    df_old, _w_old, info_old = qc_frames(df, sp, CAL_WL)          # the file's own frames (the rule before Nov 2026)
    df, oow, info = qc_frames(df, sp, CAL_WL, inc, off)
    derived = info["window_source"] != "fixed"
    cmp_rows.append(dict(file=p.name, n=len(df), in_cast=int(inc.sum()) if inc is not None else 0,
                         source=info["window_source"], ref=info["ref_source"],
                         clean_old=int(df_old["L1_clean"].sum()), clean_new=int(df["L1_clean"].sum()),
                         oow_old=int(df_old["f_outofwater"].sum()), oow_new=int(df["f_outofwater"].sum()),
                         offcast_new=int(df["f_offcast"].sum()),
                         baseline_old=int(df_old["f_baseline"].sum()), baseline_new=int(df["f_baseline"].sum()),
                         shape_old=int(df_old["f_shape"].sum()), shape_new=int(df["f_shape"].sum())))
    _cast = which_cast(t_ep, INTERVALS) if len(INTERVALS) else np.full(len(df), "", dtype=object)
    for _c in sorted(set(_cast)):
        _m = _cast == _c
        cast_rows.append(dict(cast=_c or "(between casts)", file=p.name, n=int(_m.sum()),
                              clean_old=int(df_old.loc[_m, "L1_clean"].sum()), clean_new=int(df.loc[_m, "L1_clean"].sum()),
                              oow_old=int(df_old.loc[_m, "f_outofwater"].sum()), oow_new=int(df.loc[_m, "f_outofwater"].sum())))
    FRAMES[p.name] = df; SPECTRA[p.name] = sp; REFS[p.name] = info["ref"]
    keep = (["time_utc","nitrate_uM","abs254","abs350","spec_avg","peak_counts","dark",
             "longwave_A","shape_dev","fit_rmse","T_lamp","humidity","L1_clean"] + ALL_FLAGS)
    df[keep].to_csv(L1_DIR / f"{p.stem}__L1.csv", index=False, encoding="utf-8-sig")
    row = dict(file=p.name, n=len(df), clean=int(df["L1_clean"].sum()))
    row["clean_pct"] = round(100*row["clean"]/row["n"], 1)
    for fl in ALL_FLAGS: row[fl.replace("f_","")] = int(df[fl].sum())
    inv.append(row)
    if oow is not None:
        oow_rows.append(dict(file=p.name, derived=derived, source=info["window_source"], n_frames=info["window_n"],
            reference=info["ref_source"], sides=("dark only" if info["window_source"] == "ctd" else "both"),
            peak_lo=round(oow["peak"][0]), peak_hi=round(oow["peak"][1]),
            abs254_lo=round(oow["abs254"][0], 3), abs254_hi=round(oow["abs254"][1], 3),
            specavg_lo=round(oow["spec_avg"][0]), specavg_hi=round(oow["spec_avg"][1])))
qc = pd.DataFrame(inv)
if not qc.empty:
    qc.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_L1_qc_summary.csv", index=False, encoding="utf-8-sig")
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(qc.to_string(index=False))
    print(f"\n{len(qc)} files | {int(qc['n'].sum())} frames | "
          f"{int(qc['clean'].sum())} L1-clean ({100*qc['clean'].sum()/qc['n'].sum():.1f}%)")
    owin = pd.DataFrame(oow_rows)
    owin.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_oow_windows.csv", index=False, encoding="utf-8-sig")
    print("\nderived out-of-water windows (per file):")
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(owin.to_string(index=False))
    print("L1 tables ->", L1_DIR)
    # the two rules side by side: per file, and per CTD cast (frames inside its wet interval)
    CMP = pd.DataFrame(cmp_rows); BYCAST = pd.DataFrame(cast_rows)
    CMP.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_oow_gate_comparison.csv", index=False, encoding="utf-8-sig")
    BYCAST.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_oow_gate_comparison_by_cast.csv", index=False, encoding="utf-8-sig")
    print("\nIN-WATER POPULATION: the file's own frames (old) against frames inside a CTD cast (new)")
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(CMP.to_string(index=False))
    if not BYCAST.empty:
        bc = BYCAST.groupby("cast", as_index=False)[["n", "clean_old", "clean_new", "oow_old", "oow_new"]].sum()
        print("\nper cast (frames inside the CTD's wet interval, no margin):")
        with pd.option_context("display.width", 200, "display.max_columns", 50):
            print(bc.to_string(index=False))
        ins = bc[bc["cast"] != "(between casts)"]
        if len(ins):
            print(f"\nframes inside casts: clean {int(ins['clean_old'].sum())} -> {int(ins['clean_new'].sum())} of "
                  f"{int(ins['n'].sum())}; casts with fewer clean frames under the new rule: "
                  f"{list(ins.loc[ins['clean_new'] < ins['clean_old'], 'cast']) or 'none'}")

In [ ]:
# ------------------------------------------------------------------ per-file QC figure (static PNG)
def qc_png(fname):
    df = FRAMES[fname]; t = (df["time_utc"] - df["time_utc"].iloc[0]).dt.total_seconds().to_numpy()
    good = df["L1_clean"].to_numpy()
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=(12, 7.0)); FigureCanvasAgg(fig); ax = fig.subplots(3, 1, sharex=True)
    ax[0].plot(t[good], df["nitrate_uM"].to_numpy()[good], ".", ms=1.4, color="#2E6FB0", label="L1 clean")
    ax[0].plot(t[~good], df["nitrate_uM"].to_numpy()[~good], ".", ms=1.6, color="#d62728", label="flagged")
    ax[0].axhline(0, color="#bbb", lw=0.5); ax[0].set_ylabel("NO3 (uM)")
    ax[0].legend(fontsize=7, frameon=False, markerscale=4, ncol=2)
    ax[1].plot(t, df["longwave_A"], lw=0.5, color="#8257B5"); ax[1].axhline(LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--")
    ax[1].set_ylabel("longwave |A|"); ax[1].set_ylim(0, min(0.5, max(0.12, df["longwave_A"].quantile(0.999))))
    ax[2].plot(t, df["shape_dev"], lw=0.5, color="#C77D11"); ax[2].axhline(SHAPE_DEV_HI, color="#f0a", lw=0.6, ls="--")
    ax[2].set_ylabel("shape dev"); ax[2].set_ylim(0, min(0.1, max(0.02, df["shape_dev"].quantile(0.999))))
    ax[2].set_xlabel("seconds from file start")
    for a in ax:
        for s in ("top","right"): a.spines[s].set_visible(False)
    fig.suptitle(f"{CRUISE_ID}  SUNA L1 QC  -  {fname}  ({int(good.sum())}/{len(df)} clean)", fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.97])
    out = OUT_DIR / f"{CRUISE_ID}__suna_L1__{Path(fname).stem}.png"
    fig.savefig(out, dpi=125)
    try: plt.close(fig)
    except Exception: pass
    return out

if FRAMES:
    for fn in FRAMES:
        png = qc_png(fn)
        if Image is not None:
            try: display(Image(filename=str(png)))
            except Exception: pass
        print("  saved", png.name)

In [ ]:
# ------------------------------------------------------------------ interactive: QC explorer
# Step through frames (or jump to the next flagged one) and see WHY a frame failed: raw spectrum,
# absorbance near the fit window, and the long-wave baseline. Frame title lists every flag that fired.
try:
    import ipywidgets as W
    _HAVE_W = not __import__("os").environ.get("SUNCAT_HEADLESS")   # no widgets in a headless run (tools/run_notebook.py)
except Exception:
    _HAVE_W = False
    print("ipywidgets not installed; pip install ipywidgets ipympl, then Restart Kernel + Run All.")

REF = dict(REFS)          # the reference spectrum step01 measured each file against (in-cast frames when available)

def _qc_png_bytes(fname, i):
    import io
    df = FRAMES[fname]; wl = CAL_WL; i = int(max(0, min(i, len(df) - 1)))
    s = SPECTRA[fname][i]; r = df.iloc[i]; dk = r["dark"]
    num = np.clip(s - dk, 1, None); den = np.clip(REF[fname] - dk, 1, None); A = -np.log10(num / den)
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=EXPLORER_FIGSIZE); FigureCanvasAgg(fig); ax1, ax2, ax3 = fig.subplots(1, 3)
    ax1.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10); ax1.plot(wl, s, lw=0.9, color="#333")
    ax1.axhline(dk, color="#c060c0", lw=0.7, ls="--", label=f"dark {dk:.0f}")
    ax1.set_ylim(0, 66000); ax1.set_xlabel("nm"); ax1.set_ylabel("counts"); ax1.set_title("raw", fontsize=9)
    ax1.legend(fontsize=7, frameon=False)
    m = (wl >= 205) & (wl <= 260)
    ax2.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10, label="fit 217-240")
    ax2.plot(wl[m], A[m], lw=1.2, color="#B00"); ax2.axhline(0, color="#aaa", lw=0.6)
    ax2.set_xlabel("nm"); ax2.set_ylabel("absorbance (rel)"); ax2.set_title("fit window", fontsize=9)
    ax2.legend(fontsize=7, frameon=False)
    mb = (wl >= 290) & (wl <= 370)
    ax3.axvspan(LONGWAVE_LO, LONGWAVE_HI, color="#8257B5", alpha=0.12, label="baseline 310-360")
    ax3.plot(wl[mb], A[mb], lw=1.2, color="#8257B5"); ax3.axhline(0, color="#aaa", lw=0.6)
    ax3.axhline(LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--"); ax3.axhline(-LONGWAVE_A_HI, color="#f0a", lw=0.6, ls="--")
    ax3.set_xlabel("nm"); ax3.set_ylabel("absorbance (rel)"); ax3.set_title("long-wave baseline", fontsize=9)
    ax3.legend(fontsize=7, frameon=False)
    for ax in (ax1, ax2, ax3):
        for s_ in ("top","right"): ax.spines[s_].set_visible(False)
    flags = [k[2:] for k in ALL_FLAGS if r.get(k)]
    verdict = "CLEAN" if r["L1_clean"] else "FLAGGED"
    ttl = (f"{fname}  frame {i}/{len(df)-1}  @ {r['time_utc'].strftime('%H:%M:%S')}   "
           f"NO3 {r['nitrate_uM']:.2f}   lwA {r['longwave_A']:.3f}   shape {r['shape_dev']:.4f}   [{verdict}]")
    if flags: ttl += "   " + ",".join(flags)
    fig.suptitle(ttl, fontsize=9.5, color=("#b00" if not r["L1_clean"] else "#1a7f37"))
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W and FRAMES:
    _files = list(FRAMES); _n0 = len(FRAMES[_files[0]]) - 1
    wfile  = W.Dropdown(options=_files, value=_files[0], description="file")
    wframe = W.IntSlider(min=0, max=_n0, value=0, description="frame",
                         continuous_update=False, layout=W.Layout(width="55%"))
    wnum   = W.BoundedIntText(min=0, max=_n0, value=0, description="go to", layout=W.Layout(width="150px"))
    bprev  = W.Button(description="< prev", layout=W.Layout(width="78px"))
    bnext  = W.Button(description="next >", layout=W.Layout(width="78px"))
    bflag  = W.Button(description="next flagged >>", button_style="warning", layout=W.Layout(width="140px"))
    W.link((wframe, "value"), (wnum, "value"))
    def _sync(*_):
        n = len(FRAMES[wfile.value]) - 1; wframe.max = n; wnum.max = n
        if wframe.value > n: wframe.value = 0
    wfile.observe(_sync, "value")
    bprev.on_click(lambda _: setattr(wframe, "value", max(0, wframe.value - 1)))
    bnext.on_click(lambda _: setattr(wframe, "value", min(wframe.max, wframe.value + 1)))
    def _next_flag(_):
        bad = (~FRAMES[wfile.value]["L1_clean"]).to_numpy()
        nxt = np.where(bad[wframe.value + 1:])[0]
        if len(nxt): wframe.value = int(wframe.value + 1 + nxt[0])
    bflag.on_click(_next_flag)
    def _render(fname, i): display(Image(data=_qc_png_bytes(fname, i)))
    out = W.interactive_output(_render, {"fname": wfile, "i": wframe})
    display(W.VBox([W.HBox([wfile, bprev, bnext, bflag]), W.HBox([wframe, wnum]), out]))